In [1]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train_features.csv")
test_df = pd.read_csv("../data/processed/test_features.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (226980, 31)
Test: (56746, 31)


In [2]:
X_train = train_df.drop(columns="Class")
y_train = train_df["Class"]

X_test = test_df.drop(columns="Class")
y_test = test_df["Class"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (226980, 30)
y_train: (226980,)
X_test: (56746, 30)
y_test: (56746,)


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

preprocessor = Pipeline([
    ("scaler", StandardScaler())
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed:", X_train_processed.shape)
print("X_test_processed:", X_test_processed.shape)

X_train_processed: (226980, 30)
X_test_processed: (56746, 30)


In [5]:
print("Pipeline steps:")
print(preprocessor.named_steps)

print("\nFirst 5 feature means:")
print(X_train_processed.mean(axis=0)[:5])

print("\nFirst 5 feature stds:")
print(X_train_processed.std(axis=0)[:5])

Pipeline steps:
{'scaler': StandardScaler()}

First 5 feature means:
[ 1.85320865e-17  1.56521001e-17 -1.00173441e-17  5.75997283e-18
 -2.12868561e-17]

First 5 feature stds:
[1. 1. 1. 1. 1.]


In [6]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nScaler fitted on train only:")
print(preprocessor.named_steps["scaler"].mean_[:5])

Train shape: (226980, 30)
Test shape: (56746, 30)

Scaler fitted on train only:
[-1.49258426e-16  1.03929945e-17 -1.22712465e-17  5.75997283e-18
 -2.72972626e-17]


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

final_preprocessor = Pipeline([
    ("scaler", StandardScaler())
])

X_train_final = final_preprocessor.fit_transform(X_train)
X_test_final = final_preprocessor.transform(X_test)

print(X_train_final.shape)
print(X_test_final.shape)

(226980, 30)
(56746, 30)


In [8]:
print("Number of features:", X_train.shape[1])
print("Target distribution:")
print(y_train.value_counts())

Number of features: 30
Target distribution:
Class
0    226602
1       378
Name: count, dtype: int64


In [10]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X_train,
    y_train,
    random_state=42
)

mi_scores_df = pd.DataFrame({
    "Feature": X_train.columns,
    "MI_Score": mi_scores
}).sort_values("MI_Score", ascending=False)

print(mi_scores_df)

      Feature  MI_Score
17        V17  0.008013
14        V14  0.007946
10        V10  0.007320
12        V12  0.007315
11        V11  0.006577
16        V16  0.005876
3          V3  0.004671
4          V4  0.004623
9          V9  0.004051
18        V18  0.003992
7          V7  0.003814
2          V2  0.002962
27        V27  0.002239
21        V21  0.002167
6          V6  0.002074
5          V5  0.002060
1          V1  0.001981
28        V28  0.001651
8          V8  0.001626
29  LogAmount  0.001533
0        Time  0.001432
19        V19  0.001065
20        V20  0.000787
23        V23  0.000485
24        V24  0.000335
25        V25  0.000238
22        V22  0.000164
26        V26  0.000159
15        V15  0.000103
13        V13  0.000000


In [11]:
top_features = mi_scores_df.head(20)["Feature"].tolist()

X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

print("Selected features:")
print(top_features)

print("\nTrain shape:", X_train_selected.shape)
print("Test shape:", X_test_selected.shape)

Selected features:
['V17', 'V14', 'V10', 'V12', 'V11', 'V16', 'V3', 'V4', 'V9', 'V18', 'V7', 'V2', 'V27', 'V21', 'V6', 'V5', 'V1', 'V28', 'V8', 'LogAmount']

Train shape: (226980, 20)
Test shape: (56746, 20)


In [13]:
missing_train = X_train_selected.isnull().sum().sum()
missing_test = X_test_selected.isnull().sum().sum()

print("Missing values - train:", missing_train)
print("Missing values - test:", missing_test)
print("Feature count:", len(top_features))

Missing values - train: 0
Missing values - test: 0
Feature count: 20


In [14]:
print("Final train:", X_train_final.shape)
print("Final test:", X_test_final.shape)

print("\nTarget train:")
print(y_train.value_counts())

print("\nTarget test:")
print(y_test.value_counts())

Final train: (226980, 30)
Final test: (56746, 30)

Target train:
Class
0    226602
1       378
Name: count, dtype: int64

Target test:
Class
0    56651
1       95
Name: count, dtype: int64


In [15]:
X_train_final = X_train_selected.copy()
X_test_final = X_test_selected.copy()

print("Final train:", X_train_final.shape)
print("Final test:", X_test_final.shape)

Final train: (226980, 20)
Final test: (56746, 20)


In [16]:
train_final_df = X_train_final.copy()
train_final_df["Class"] = y_train.values

test_final_df = X_test_final.copy()
test_final_df["Class"] = y_test.values

train_final_df.to_csv("../data/processed/train_final.csv", index=False)
test_final_df.to_csv("../data/processed/test_final.csv", index=False)

print(train_final_df.shape)
print(test_final_df.shape)

(226980, 21)
(56746, 21)


In [17]:
import os

print("Train file exists:", os.path.exists("../data/processed/train_final.csv"))
print("Test file exists:", os.path.exists("../data/processed/test_final.csv"))

Train file exists: True
Test file exists: True
